# grid_to_df
notebook for converting garstec out hdf5 files to neural network friendly dataframes saved back to hdf5

In [1]:
#misc
import pandas as pd
import numpy as np
import h5py
import os

In [2]:
np.eye(4)

array([[1., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 1.]])

## import garstec h5 file

In [2]:
garstec_h5 = h5py.File("../grids/Garstec_AS09_chiara.hdf5", 'r') ### EDIT WITH YOUR FILEPATH

print(list(garstec_h5['grid/tracks/track00001']))

['BP_GAIA', 'FeH', 'FeHini', 'G_GAIA', 'LPhot', 'MMaxNucE', 'Mbcz', 'Mcore', 'McoreX', 'MeH', 'MeHini', 'PS', 'RMaxNucE', 'RP_GAIA', 'Rbcz', 'Rcore', 'RcoreX', 'TAMS', 'Teff', 'ZAMSLPhot', 'ZAMSTeff', 'age', 'alphaFe', 'alphaMLT', 'd02fit', 'd02mean', 'dage', 'dif', 'dnuAsf', 'dnuSer', 'dnufit', 'dnufitMos12', 'dnuscal', 'epsfit', 'epsfitMos12', 'errflagSer', 'eta', 'fdnuAsf', 'fdnuSer', 'gcut', 'logg', 'massfin', 'massini', 'modnum', 'name', 'numax', 'numaxAsf', 'nummodSer', 'osc', 'osckey', 'ove', 'radPhot', 'radTot', 'rho', 'rhocen', 'tau0', 'taubcz', 'tauhe', 'volume_weight', 'xcen', 'xini', 'xsur', 'ycen', 'yini', 'ysur', 'zcen', 'zini', 'zsur']


## def and run track_df_gen func

In [3]:
def track_df_no_modes(
    grid: h5py.File,
    track_ids: list[str, ...],
    headers: list[str, str, str] = ["age", "Teff", "LPhot"],
):
    i = 0
    for track_id in track_ids:
        track = grid["grid/tracks/track" + track_id]

        track_array = np.full(len(track[headers[0]]), int(track_id))

        for header in headers:
            track_array = np.column_stack((track_array, np.array(track[header])))

        if i == 0:
            tracks_array = track_array
            i = 1
        else:
            tracks_array = np.vstack((tracks_array, track_array))

    return pd.DataFrame(tracks_array, columns=["track_id"] + headers)

def track_df_gen(grid, track_ids, n_min=15, n_max=25, headers=['age', 'Teff', 'LPhot']):
    """
    Convert garstec hdf5 file (grid) to NN friendly dataframe file for training.
    
    args:
    grid -- garstec hdf5 out file
    track_ids -- list of track ids (including leading zeros) to iterate over

    kwargs:
    n_min -- lower limit of radial order to load/pad
    n_max -- upper limit of radial order to load/pad
    headers -- relevant headers to include in final dataframe (inputs and outputs for training, typically)

    returns:
    pandas dataframe with columns:
        - ['track_id'] -- integer track id from GARSTEC hdf5 out file, no leading 0s
        - [headers] -- columns for each header defined by 'headers' kwarg
        - [nu_headers] -- columns for mode frequency values for radial orders in range n_min->n_max
    
    notes:
    - Operates track by track, and fills missing mode freqencies by padding using GARSTEC dnufit.
    - Modes loaded/filled between a min and max radial order. This must be consistent between all dataframe rows.
    - Warning is printed when padding is occuring. This is unexpected behaviour if radial orders between n_min and n_max should be present for all points in the grid.
    """
    nu_headers = [f"nu_0_{n}" for n in range(n_min, n_max+1)]
    first_switch=0
    for track_id in track_ids:
        print(str(track_id), end="\r")
        track = grid['grid/tracks/track'+track_id]
        
        track_array = np.full(len(track[headers[0]]), int(track_id))

        for header in headers:
            track_array = np.column_stack((track_array, np.array(track[header])))
        
        age = track['age']
    
        points = np.array(age) #np.array(age)[np.where(np.array(age)<=max_age)[0]]
        first_point = points[0]
        last_point = points[-1]
        
        osckeys = track['osckey']
        dnufits = track['dnufit']

        i = 0
        padding_switch = 0
        for point in points:
            n_vals = osckeys[i][1][np.where(osckeys[i][0] == 0)[0]]
            n_upper = n_vals[-1]
            n_lower = n_vals[0]
            dnufit = dnufits[i]
            
            nu_vals = list(track['osc'][i][0][np.where(osckeys[i][0] == 0)])
            if n_upper < n_max:
                nu_max = nu_vals[-1]
                n_diff = n_max - n_upper
                try:
                    nu_upper_pad = (np.linspace(1,n_diff,n_diff)*dnufit)+nu_max
                except:
                    nu_upper_pad = np.full(n_diff, nu_max)
                nu_vals = nu_vals + nu_upper_pad.tolist()
                n_upper=n_max
                padding_switch = 1

            
            if n_lower > n_min:
                nu_min = nu_vals[0]
                n_diff = n_lower - n_min
                try:
                    nu_lower_pad = nu_min - (np.linspace(n_diff,1,n_diff)*dnufit)
                except:
                    nu_lower_pad = np.full(n_diff, nu_min)
                nu_vals = nu_lower_pad.tolist() + nu_vals
                n_lower = n_min
                padding_switch = 1
            
            nu_vals = nu_vals[n_min - n_lower:(n_max - n_lower)+1]
            
            if i == 0:
                nu_vals_arr = nu_vals
            else:
                nu_vals_arr = np.vstack((nu_vals_arr, nu_vals))
            i+=1
        track_array = np.concatenate((track_array, nu_vals_arr), axis=1)
       
        if first_switch == 0:
            tracks_array = track_array
            first_switch = 1
        else:
            tracks_array = np.vstack((tracks_array, track_array))

        if padding_switch == 1:
            print(f"padding! requested radial order range {n_min}->{n_max} exceeded range present in track{track_id}")


    return pd.DataFrame(tracks_array, columns = ['track_id']+headers+nu_headers)

headers = ['massini', 'zini', 'yini', 'alphaMLT', 'alphaFe', 'eta', 'age', 'TAMS', 'logg', 'LPhot', 'Teff', 'FeH', 'numax', 'dnuSer']

track_ids = [track_name.replace('track', '') for track_name in list(garstec_h5['grid/tracks'])]

garstec_df = track_df_no_modes(garstec_h5,track_ids, headers=headers)

## check df

In [4]:
garstec_df.describe()

,track_id,massini,zini,yini,alphaMLT,alphaFe,eta,age,TAMS,logg,LPhot,Teff,FeH,numax,dnuSer
count,7.483746e+06,7.483746e+06,7.483746e+06,7.483746e+06,7.483746e+06,7.483746e+06,7.483746e+06,7.483746e+06,7.483746e+06,7.483746e+06,7.483746e+06,7.483746e+06,7.483746e+06,7.483746e+06,7.483746e+06
mean,4.995407e+03,1.125641e+00,7.081989e-03,2.824216e-01,1.916603e+00,2.083742e-01,1.498787e-01,6.377355e+03,1.067388e+00,2.531390e+00,6.364577e+01,4.951428e+03,-7.783541e-01,2.329383e-02,4.911477e-02
std,2.883893e+03,2.193117e-01,8.982759e-03,3.725388e-02,2.299017e-01,2.579447e-01,8.657509e-02,4.615981e+03,3.882874e-02,4.186764e-01,4.328946e+01,4.101017e+02,6.133893e-01,3.297786e-02,5.061614e-02
min,1.000000e+00,7.000000e-01,9.149300e-05,2.200159e-01,1.500049e+00,-2.000000e-01,3.662109e-05,8.274848e+02,1.000000e+00,1.944352e+00,1.868248e+00,3.787317e+03,-1.999866e+00,3.397339e-03,1.414107e-02
25%,2.502000e+03,9.440000e-01,9.819730e-04,2.500879e-01,1.720117e+00,0.000000e+00,7.485352e-02,2.752553e+03,1.039768e+00,2.194846e+00,2.809932e+01,4.684124e+03,-1.271780e+00,6.274900e-03,1.969099e-02
50%,4.999000e+03,1.129000e+00,3.382367e-03,2.811438e-01,1.924316e+00,2.000000e-01,1.499268e-01,4.822933e+03,1.056742e+00,2.424475e+00,5.587388e+01,4.953846e+03,-7.265381e-01,1.059033e-02,2.934707e-02
75%,7.482000e+03,1.314000e+00,9.776608e-03,3.141516e-01,2.117188e+00,4.000000e-01,2.249084e-01,8.881607e+03,1.086883e+00,2.777743e+00,9.166179e+01,5.180804e+03,-2.555826e-01,2.341717e-02,5.458154e-02
max,1.000000e+04,1.500000e+00,5.985032e-02,3.499921e-01,2.299902e+00,6.000000e-01,2.999634e-01,2.000000e+04,1.236157e+00,3.790482e+00,2.508199e+02,9.936561e+03,2.169955e-01,2.473394e-01,3.027724e-01


In [8]:
garstec_df.to_hdf('../grids/Chiara.hdf5', key = 'df')